## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [2]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [3]:
MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [4]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [19]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

### These LangChain objects implement the method `invoke()`

In [20]:
retriever.invoke("Who is priya?")

[Document(id='8ba9538f-31ad-4c49-a985-b887adc99b17', metadata={'doc_type': 'employees', 'source': 'DALL_FULL_MD_Dataset\\employees\\emp_002 priya sharma.md'}, page_content='# Employee Profile: Priya Sharma\nRole: Memory Systems Engineer\nSkills: DDR5, HBM, Firmware validation, Python testing\nCareer: Joined in 2020, key contributor to HBM stack'),
 Document(id='641e365e-28ee-4948-b56d-b3f14eb08bc2', metadata={'source': 'DALL_FULL_MD_Dataset\\employees\\emp_008 ananya roy.md', 'doc_type': 'employees'}, page_content='# Employee Profile: Ananya Roy\nRole: Hardware Validation Engineer\nSkills: Benchmarking, Power analysis, Linux\nCareer: Joined in 2022, AI server testing'),
 Document(id='c90fae23-232f-406b-86eb-4979fce46661', metadata={'source': 'DALL_FULL_MD_Dataset\\employees\\emp_004 sara khan.md', 'doc_type': 'employees'}, page_content='# Employee Profile: Sara Khan\nRole: QA Lead\nSkills: Hardware testing, Stress testing, Compliance\nCareer: Joined in 2018, leads QA automation'),
 Doc

In [22]:
llm.invoke("Who is priya?")

AIMessage(content='"Priya" is a common given name in India and other South Asian countries, meaning "beloved" or "dear" in Sanskrit. Without additional context, it\'s difficult to identify a specific individual named Priya. Could you please provide more details or specify which Priya you are referring to?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 12, 'total_tokens': 73, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_44848d1550', 'id': 'chatcmpl-D5vWp8wCK9GuZhuLl0tjVehCisSrS', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--ab1cb94f-3052-4a22-b0b4-ca028df7b69b-0', usage_metadata={'input_tokens': 12, 'output_tokens': 61, 'total_tok

## Time to put this together!

In [11]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company DALL Technology.
You are chatting with a user about DALL Technology.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

# RAG


In [13]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [17]:
answer_question("how many contract happened", [])

'Based on the information provided, there are three contracts mentioned:\n\n1. Government Infrastructure Contract (Duration: 5 Years)\n2. Data Center Partnership Contract (Duration not specified)\n3. Enterprise Supply Contract (Duration: 3 Years)\n4. Annual Maintenance Contract (Duration not specified)\n\nSo, in total, there are four contracts.'

## What could possibly come next? 😂

In [18]:
gr.ChatInterface(answer_question).launch()

c:\Users\asus\projects\github_push_llm\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!